# M2a · Staging profile (CRISP-DM Phase 2 replay)

**Goal:** understand the shape, volume, and freshness of the five source tables we will lift into the warehouse.

No writes.

**Exit criterion:** you can answer — for each source table — (row count, time column min/max, null rate of key columns).


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
SOURCES = [
    # (staging table, event-time column, target fact)
    ("path",               "begin_path_time",    "fact_trip"),
    ("rep_overspeed",      "begin_path_time",    "fact_overspeed"),
    ("stop",               "stop_start",         "fact_stop"),
    ("notification",       "created_at",         "fact_speed_notification"),
    ("rep_activity_daily", "activity_start_time","fact_daily_activity"),
]


## 3. Execute — per-source profile

In [ ]:
import pandas as pd
rows = []
with get_engine().connect() as conn:
    for tbl, ts, fact in SOURCES:
        q = text(f"""
            SELECT COUNT(*) AS n,
                   MIN({ts}) AS min_ts,
                   MAX({ts}) AS max_ts,
                   COUNT(*) FILTER (WHERE {ts} IS NULL) AS null_ts,
                   COUNT(DISTINCT tenant_id) AS tenants,
                   COUNT(DISTINCT device_id) AS devices
            FROM staging.{tbl}
        """)
        r = conn.execute(q).mappings().first()
        rows.append(dict(r, table=tbl, event_time_col=ts, target_fact=fact))
profile = pd.DataFrame(rows)[['table','target_fact','event_time_col','n','min_ts','max_ts','null_ts','tenants','devices']]
profile


## 4. Inspect — volume chart

In [ ]:
import matplotlib.pyplot as plt
ax = profile.plot.bar(x='table', y='n', legend=False, figsize=(8,3))
ax.set_ylabel('rows')
ax.set_title('Staging row counts per source')
plt.tight_layout(); plt.show()
